# Hindi Fake News — Final Ensemble

combining Sarvam-1 (7B LoRA) and MuRIL-base predictions using 4 different ensemble strategies.

**input**: soft probabilities + sample IDs dumped by the individual model notebooks  
**dataset**: CONSTRAINT 2021 hindi fake news  

**ensemble strategies tried:**
1. simple average of calibrated probs  
2. AUC-weighted average  
3. majority voting with veto (MV-V) — from Praseed et al. 2023  
4. stacking with logistic regression meta-learner  

all strategies go through Platt calibration first (fit on val, applied everywhere).  
best threshold tuned on val for each strategy, then applied to test.

note: both models don't process identical samples due to different tokenizer-level drops.  
using `np.intersect1d` on saved sample IDs to align them before combining.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    matthews_corrcoef, classification_report,
    roc_auc_score, confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/datasets/sherwinmazarello/hindi-fakenews-ensemble-prob"

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(
        f"folder not found: {DATA_DIR}\n"
        f"available: {os.listdir('/kaggle/input')}"
    )

all_files = os.listdir(DATA_DIR)
print(f"found {len(all_files)} files in {DATA_DIR}")
print(sorted(all_files))


In [ ]:
# helper to find files by keyword without hardcoding exact names
def find_file(keywords_must, keywords_must_not=[]):
    matches = [
        f for f in all_files
        if f.endswith(".npy")
        and all(k in f.lower() for k in keywords_must)
        and not any(k in f.lower() for k in keywords_must_not)
    ]
    if not matches:
        raise FileNotFoundError(
            f"no file with keywords {keywords_must} (excl {keywords_must_not})\n"
            f"files: {sorted(all_files)}"
        )
    matches.sort(key=len)   # prefer shortest name (least suffix noise)
    return os.path.join(DATA_DIR, matches[0])

def load(keywords_must, keywords_must_not=[]):
    path = find_file(keywords_must, keywords_must_not)
    arr  = np.load(path)
    print(f"  loaded {os.path.basename(path):<50s}  shape={arr.shape}")
    return arr

print("loading all 18 files...")

# sarvam
s_train_p  = load(["sarvam", "train", "prob"])
s_valid_p  = load(["sarvam", "valid", "prob"])
s_test_p   = load(["sarvam", "test",  "prob"])
s_train_l  = load(["sarvam", "train", "label"])
s_valid_l  = load(["sarvam", "valid", "label"])
s_test_l   = load(["sarvam", "test",  "label"])
s_train_id = load(["sarvam", "train", "id"])
s_valid_id = load(["sarvam", "valid", "id"])
s_test_id  = load(["sarvam", "test",  "id"])

# muril
m_train_p  = load(["muril", "train", "prob"])
m_valid_p  = load(["muril", "valid", "prob"])
m_test_p   = load(["muril", "test",  "prob"])
m_train_l  = load(["train", "label"], keywords_must_not=["sarvam", "muril"])
m_valid_l  = load(["valid", "label"], keywords_must_not=["sarvam", "muril"])
m_test_l   = load(["test",  "label"], keywords_must_not=["sarvam", "muril"])
m_train_id = load(["muril", "train", "id"])
m_valid_id = load(["muril", "valid", "id"])
m_test_id  = load(["muril", "test",  "id"])

print("all 18 files loaded")


In [ ]:
# align both models to shared sample IDs
# each model might have dropped different rows during preprocessing,
# so we intersect on sample IDs to compare only truly shared articles

def align(sp, sl, si, mp, ml, mi, name):
    shared = np.intersect1d(si, mi)
    sm     = np.isin(si, shared)
    mm     = np.isin(mi, shared)
    so     = np.argsort(si[sm])
    mo     = np.argsort(mi[mm])
    print(f"  {name}: sarvam={len(si)}  muril={len(mi)}  shared={len(shared)}"
          f"  (dropped {len(si)-len(shared)} sarvam, {len(mi)-len(shared)} muril)")
    return sp[sm][so], sl[sm][so], mp[mm][mo], ml[mm][mo]

print("aligning splits to shared sample IDs...")
tr_sp, tr_sl, tr_mp, tr_ml = align(s_train_p, s_train_l, s_train_id, m_train_p, m_train_l, m_train_id, "Train")
va_sp, va_sl, va_mp, va_ml = align(s_valid_p, s_valid_l, s_valid_id, m_valid_p, m_valid_l, m_valid_id, "Valid")
te_sp, te_sl, te_mp, te_ml = align(s_test_p,  s_test_l,  s_test_id,  m_test_p,  m_test_l,  m_test_id,  "Test")

# sanity check — same articles should have same labels
assert np.array_equal(va_sl, va_ml), "label mismatch on valid set"
assert np.array_equal(te_sl, te_ml), "label mismatch on test set"

y_train = tr_sl
y_valid = va_sl
y_test  = te_sl
print("label alignment verified")


In [ ]:
# Platt calibration — fit a logistic regression on val probs to fix overconfidence
# fit once per model on val, then apply to all splits

def calibrate(valid_p, valid_l, arrays):
    lr = LogisticRegression(C=1.0, max_iter=1000)
    lr.fit(valid_p[:, 1].reshape(-1, 1), valid_l)
    return [lr.predict_proba(a[:, 1].reshape(-1, 1)) for a in arrays]

print("Platt calibrating both models...")
s_tr_c, s_va_c, s_te_c = calibrate(va_sp, y_valid, [tr_sp, va_sp, te_sp])
m_tr_c, m_va_c, m_te_c = calibrate(va_mp, y_valid, [tr_mp, va_mp, te_mp])
print("calibration done")


In [ ]:
# helper functions for threshold search + metric computation

def best_threshold(probs, labels):
    """find threshold that maximises macro-F1 on the given split."""
    bt, bs = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        s = f1_score(labels, (probs >= t).astype(int), average="macro", zero_division=0)
        if s > bs:
            bs, bt = s, t
    return bt

def get_metrics(probs, labels, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "preds": preds,
        "acc":   accuracy_score(labels, preds),
        "prec":  precision_score(labels, preds, average="macro", zero_division=0),
        "rec":   recall_score(labels, preds,    average="macro", zero_division=0),
        "f1":    f1_score(labels, preds, average="binary", zero_division=0),
        "mf1":   f1_score(labels, preds, average="macro",  zero_division=0),
        "mcc":   matthews_corrcoef(labels, preds),
        "auc":   roc_auc_score(labels, probs),
    }

def print_metrics(name, m, labels):
    print(f"\n{name}")
    print(f"  acc={m['acc']:.4f}  prec={m['prec']:.4f}  rec={m['rec']:.4f}")
    print(f"  bin-F1={m['f1']:.4f}  macro-F1={m['mf1']:.4f}  MCC={m['mcc']:.4f}  AUC={m['auc']:.4f}")
    print(classification_report(labels, m["preds"], target_names=["Real", "Fake"], digits=4))


In [ ]:
# individual model results for comparison baseline
print("individual model results (own full test sets):")

t_s = best_threshold(s_test_p[:, 1], s_test_l)
t_m = best_threshold(m_test_p[:, 1], m_test_l)
r_s = get_metrics(s_test_p[:, 1], s_test_l, t_s)
r_m = get_metrics(m_test_p[:, 1], m_test_l, t_m)
print_metrics(f"Sarvam-1 alone (t={t_s:.2f})", r_s, s_test_l)
print_metrics(f"MuRIL alone    (t={t_m:.2f})", r_m, m_test_l)


## Ensemble strategies

**A: Simple average** — equal weight to both models after Platt calibration  
**B: AUC-weighted average** — weight by each model's val AUC  
**C: MV-V (majority voting with veto)** — from Praseed et al. 2023. High-confidence model overrides when the other is uncertain  
**D: Stacking** — logistic regression meta-learner trained on val calibrated probs  

Threshold tuned on val for A, B, D. Theta tuned on val for C.


In [ ]:
# AUC-based weights from val set
auc_s = roc_auc_score(y_valid, va_sp[:, 1])
auc_m = roc_auc_score(y_valid, va_mp[:, 1])
w_s   = auc_s / (auc_s + auc_m)
w_m   = 1 - w_s
print(f"AUC weights: sarvam={w_s:.3f}  muril={w_m:.3f}")

# A: simple average
avg_va = (s_va_c[:, 1] + m_va_c[:, 1]) / 2.0
avg_te = (s_te_c[:, 1] + m_te_c[:, 1]) / 2.0
avg_tr = (s_tr_c[:, 1] + m_tr_c[:, 1]) / 2.0
t_avg  = best_threshold(avg_va, y_valid)
r_avg_tr = get_metrics(avg_tr, y_train, t_avg)
r_avg_va = get_metrics(avg_va, y_valid, t_avg)
r_avg_te = get_metrics(avg_te, y_test,  t_avg)

# B: weighted average
wtd_va = w_s * s_va_c[:, 1] + w_m * m_va_c[:, 1]
wtd_te = w_s * s_te_c[:, 1] + w_m * m_te_c[:, 1]
wtd_tr = w_s * s_tr_c[:, 1] + w_m * m_tr_c[:, 1]
t_wtd  = best_threshold(wtd_va, y_valid)
r_wtd_tr = get_metrics(wtd_tr, y_train, t_wtd)
r_wtd_va = get_metrics(wtd_va, y_valid, t_wtd)
r_wtd_te = get_metrics(wtd_te, y_test,  t_wtd)

print("A and B done")


In [ ]:
# C: majority voting with veto (Praseed et al. 2023)
# if one model is confident (>= theta) and the other isn't, defer to the confident one
# otherwise take simple average

def mvv_predict(ps_arr, pm_arr, theta):
    preds = []
    for ps, pm in zip(ps_arr, pm_arr):
        sc = max(ps, 1 - ps)
        mc = max(pm, 1 - pm)
        if   sc >= theta and mc < theta: preds.append(int(ps >= 0.5))
        elif mc >= theta and sc < theta: preds.append(int(pm >= 0.5))
        else:                            preds.append(int((ps + pm) / 2 >= 0.5))
    return np.array(preds)

# tune theta on val
best_theta, best_theta_f1 = 0.5, 0.0
for theta in np.arange(0.5, 1.0, 0.02):
    preds = mvv_predict(s_va_c[:, 1], m_va_c[:, 1], theta)
    score = f1_score(y_valid, preds, average="macro", zero_division=0)
    if score > best_theta_f1:
        best_theta_f1, best_theta = score, theta
print(f"MV-V best theta={best_theta:.2f}  val macro-F1={best_theta_f1:.4f}")

def mvv_metrics(ps_arr, pm_arr, labels, theta):
    preds = mvv_predict(ps_arr, pm_arr, theta)
    probs = (ps_arr + pm_arr) / 2.0
    base  = get_metrics(probs, labels, 0.5)
    return base | {
        "preds": preds,
        "acc":  accuracy_score(labels, preds),
        "prec": precision_score(labels, preds, average="macro", zero_division=0),
        "rec":  recall_score(labels, preds,    average="macro", zero_division=0),
        "f1":   f1_score(labels, preds, average="binary", zero_division=0),
        "mf1":  f1_score(labels, preds, average="macro",  zero_division=0),
        "mcc":  matthews_corrcoef(labels, preds),
    }

r_mvv_tr = mvv_metrics(s_tr_c[:, 1], m_tr_c[:, 1], y_train, best_theta)
r_mvv_va = mvv_metrics(s_va_c[:, 1], m_va_c[:, 1], y_valid, best_theta)
r_mvv_te = mvv_metrics(s_te_c[:, 1], m_te_c[:, 1], y_test,  best_theta)


In [ ]:
# D: stacking — logistic regression meta-learner
# train on val set calibrated probs from both models
meta = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced")
meta.fit(np.hstack([s_va_c, m_va_c]), y_valid)

st_te_p = meta.predict_proba(np.hstack([s_te_c, m_te_c]))[:, 1]
st_va_p = meta.predict_proba(np.hstack([s_va_c, m_va_c]))[:, 1]
st_tr_p = meta.predict_proba(np.hstack([s_tr_c, m_tr_c]))[:, 1]
t_st    = best_threshold(st_va_p, y_valid)
r_st_tr = get_metrics(st_tr_p, y_train, t_st)
r_st_va = get_metrics(st_va_p, y_valid, t_st)
r_st_te = get_metrics(st_te_p, y_test,  t_st)

print("stacking done")


In [ ]:
# print all ensemble results side by side
print("\nensemble results — all strategies — all splits")
print("=" * 70)

for strategy, tr, va, te in [
    (f"A) Simple Average         (t={t_avg:.2f})",                r_avg_tr, r_avg_va, r_avg_te),
    (f"B) Weighted Avg S={w_s:.2f},M={w_m:.2f} (t={t_wtd:.2f})", r_wtd_tr, r_wtd_va, r_wtd_te),
    (f"C) MV-V (theta={best_theta:.2f})",                         r_mvv_tr, r_mvv_va, r_mvv_te),
    (f"D) Stacking LogReg        (t={t_st:.2f})",                 r_st_tr,  r_st_va,  r_st_te),
]:
    print(f"\n  {strategy}")
    print(f"  {'split':<12} {'acc':>7} {'prec':>7} {'rec':>7} {'bin-F1':>8} {'mac-F1':>8} {'MCC':>7} {'AUC':>7}")
    print("  " + "-" * 65)
    for sname, met in [("Train", tr), ("Val", va), ("Test", te)]:
        print(f"  {sname:<12} {met['acc']:>7.4f} {met['prec']:>7.4f} "
              f"{met['rec']:>7.4f} {met['f1']:>8.4f} {met['mf1']:>8.4f} "
              f"{met['mcc']:>7.4f} {met['auc']:>7.4f}")


In [ ]:
# confusion matrices for best ensemble (weighted average) on all 3 splits
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Weighted Avg Ensemble — Confusion Matrices", fontsize=13, y=1.02)

for ax, (name, y_t, y_p) in zip(axes, [
    ("Train", y_train, r_wtd_tr["preds"]),
    ("Val",   y_valid, r_wtd_va["preds"]),
    ("Test",  y_test,  r_wtd_te["preds"]),
]):
    cm = confusion_matrix(y_t, y_p)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"], cbar=False)
    ax.set_title(f"{name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()
